# 💧 UFRJ HydroSense: Pipeline Independente ERA5-Land

Este notebook é uma ferramenta independente para download e processamento de dados meteorológicos do **ERA5-Land**. Ele gera o input necessário para o modelo de previsão de vazão sem depender de outros scripts do repositório.

### 📋 Requisitos
Instalar as dependências necessárias:
```bash
pip install pandas cdsapi
```

In [ ]:
import os
import pandas as pd
import cdsapi
import zipfile
import io
import glob
from concurrent.futures import ThreadPoolExecutor, as_completed

# --- CONFIGURAÇÃO DINÂMICA ---
CDS_API_KEY = "SUA-CHAVE-AQUI"  # Cole sua API Key do Copernicus aqui
CDS_API_URL = "https://cds-beta.climate.copernicus.eu/api"

STATION_ID = "19098"  # Opções: "19098", "58350001", "19094"
DATE_RANGE = "2024-01-01/2025-07-29"  # Período de interesse

# --- MAPEAMENTO DE COORDENADAS (ESTAÇÕES ATIVAS) ---
STATION_COORDS = {
    "19098": [
        {"Subbasin": 2, "Lat": -22.79, "Lon": -44.21},
        {"Subbasin": 3, "Lat": -22.83, "Lon": -44.20},
        {"Subbasin": 4, "Lat": -22.79, "Lon": -44.17},
        {"Subbasin": 5, "Lat": -22.75, "Lon": -44.13}
    ],
    "58350001": [
        {"Subbasin": 1, "Lat": -22.76, "Lon": -44.09},
        {"Subbasin": 2, "Lat": -22.79, "Lon": -44.21},
        {"Subbasin": 3, "Lat": -22.83, "Lon": -44.20},
        {"Subbasin": 4, "Lat": -22.79, "Lon": -44.17},
        {"Subbasin": 5, "Lat": -22.75, "Lon": -44.13},
        {"Subbasin": 6, "Lat": -22.78, "Lon": -44.10},
        {"Subbasin": 7, "Lat": -22.72, "Lon": -44.09},
        {"Subbasin": 8, "Lat": -22.67, "Lon": -43.96}
    ],
    "19094": [
        {"Subbasin": 1, "Lat": -22.36, "Lon": -44.45}, {"Subbasin": 2, "Lat": -22.38, "Lon": -44.13}, 
        {"Subbasin": 3, "Lat": -22.43, "Lon": -44.31}, {"Subbasin": 4, "Lat": -22.56, "Lon": -44.68},
        {"Subbasin": 5, "Lat": -22.51, "Lon": -44.55}, {"Subbasin": 6, "Lat": -22.48, "Lon": -44.45},
        {"Subbasin": 7, "Lat": -22.45, "Lon": -43.91}, {"Subbasin": 8, "Lat": -22.52, "Lon": -44.03},
        {"Subbasin": 9, "Lat": -22.52, "Lon": -44.86}, {"Subbasin": 10, "Lat": -22.50, "Lon": -44.39},
        {"Subbasin": 11, "Lat": -22.63, "Lon": -44.42}, {"Subbasin": 12, "Lat": -22.56, "Lon": -45.10},
        {"Subbasin": 13, "Lat": -22.57, "Lon": -44.13}, {"Subbasin": 14, "Lat": -22.51, "Lon": -44.23},
        {"Subbasin": 15, "Lat": -22.56, "Lon": -44.97}, {"Subbasin": 16, "Lat": -22.52, "Lon": -44.58},
        {"Subbasin": 17, "Lat": -22.66, "Lon": -44.30}, {"Subbasin": 18, "Lat": -22.65, "Lon": -44.83},
        {"Subbasin": 19, "Lat": -22.72, "Lon": -45.14}, {"Subbasin": 20, "Lat": -22.65, "Lon": -44.97},
        {"Subbasin": 21, "Lat": -22.75, "Lon": -44.89}, {"Subbasin": 22, "Lat": -22.66, "Lon": -45.01},
        {"Subbasin": 23, "Lat": -22.87, "Lon": -45.34}, {"Subbasin": 24, "Lat": -22.87, "Lon": -44.85},
        {"Subbasin": 25, "Lat": -23.06, "Lon": -45.69}, {"Subbasin": 26, "Lat": -22.98, "Lon": -45.98},
        {"Subbasin": 27, "Lat": -22.97, "Lon": -45.83}, {"Subbasin": 28, "Lat": -22.90, "Lon": -45.51},
        {"Subbasin": 29, "Lat": -22.99, "Lon": -45.12}, {"Subbasin": 30, "Lat": -23.10, "Lon": -45.46},
        {"Subbasin": 31, "Lat": -23.07, "Lon": -44.93}, {"Subbasin": 32, "Lat": -23.08, "Lon": -46.12},
        {"Subbasin": 33, "Lat": -23.14, "Lon": -45.19}, {"Subbasin": 34, "Lat": -23.27, "Lon": -45.45},
        {"Subbasin": 35, "Lat": -23.15, "Lon": -46.06}, {"Subbasin": 36, "Lat": -23.23, "Lon": -45.02},
        {"Subbasin": 37, "Lat": -23.18, "Lon": -45.97}, {"Subbasin": 38, "Lat": -23.21, "Lon": -46.07},
        {"Subbasin": 39, "Lat": -23.27, "Lon": -46.25}, {"Subbasin": 40, "Lat": -23.16, "Lon": -45.90},
        {"Subbasin": 41, "Lat": -23.25, "Lon": -45.65}, {"Subbasin": 42, "Lat": -23.26, "Lon": -45.93},
        {"Subbasin": 43, "Lat": -23.26, "Lon": -45.23}, {"Subbasin": 44, "Lat": -23.21, "Lon": -46.03},
        {"Subbasin": 45, "Lat": -23.37, "Lon": -46.16}, {"Subbasin": 46, "Lat": -23.38, "Lon": -45.72},
        {"Subbasin": 47, "Lat": -23.37, "Lon": -45.82}, {"Subbasin": 48, "Lat": -23.40, "Lon": -45.33},
        {"Subbasin": 49, "Lat": -23.35, "Lon": -45.89}, {"Subbasin": 50, "Lat": -23.41, "Lon": -46.05},
        {"Subbasin": 51, "Lat": -23.44, "Lon": -45.95}, {"Subbasin": 52, "Lat": -23.40, "Lon": -45.58},
        {"Subbasin": 53, "Lat": -23.46, "Lon": -45.66}, {"Subbasin": 54, "Lat": -23.53, "Lon": -45.54}
    ]
}

# Variáveis a baixar
VARIABLES = [
    "2m_dewpoint_temperature",
    "surface_solar_radiation_downwards",
    "surface_thermal_radiation_downwards",
    "2m_temperature",
    "total_precipitation",
    "10m_u_component_of_wind",
    "10m_v_component_of_wind",
]

# Diretórios (serão criados localmente)
RAW_DIR = "data_raw_era5"
OUTPUT_DIR = "data_inference"
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 1. Download de Dados (Parallel Request)
Executa as requisições para o Copernicus Climate Data Store.

In [ ]:
def download_subbasin(subid, lat, lon):
    filename = os.path.join(RAW_DIR, f"era5_{subid}_{lat}_{lon}.csv.zip")
    
    if os.path.exists(filename):
        print(f"-> Sub-bacia {subid} já existe localmente.")
        return subid, True

    # Configuração dinâmica do cliente sem .cdsapirc
    client = cdsapi.Client(url=CDS_API_URL, key=CDS_API_KEY)
    
    request = {
        "variable": VARIABLES,
        "location": {"longitude": lon, "latitude": lat},
        "date": [DATE_RANGE],
        "data_format": "csv"
    }
    
    try:
        print(f"-> Solicitando Sub-bacia {subid} (Lat: {lat}, Lon: {lon})...")
        client.retrieve("reanalysis-era5-land-timeseries", request, filename)
        return subid, True
    except Exception as e:
        print(f"!! Erro na Sub-bacia {subid}: {e}")
        return subid, False

coords_list = STATION_COORDS.get(STATION_ID, [])
if not coords_list:
    print(f"Erro: STATION_ID {STATION_ID} não encontrado no mapeamento.")
else:
    print(f"Iniciando download para {len(coords_list)} pontos da Estação {STATION_ID}...")
    with ThreadPoolExecutor(max_workers=5) as executor:
        futures = {
            executor.submit(download_subbasin, item['Subbasin'], item['Lat'], item['Lon']): item
            for item in coords_list
        }
        for future in as_completed(futures):
            subid, success = future.result()
            # Status impresso pela função

## 2. Processamento e Agregação
Lê os arquivos baixados, consolida e transforma de horário para diário.

In [ ]:
dfs_list = []
for item in coords_list:
    subid, lat, lon = item['Subbasin'], item['Lat'], item['Lon']
    pattern = os.path.join(RAW_DIR, f"era5_{subid}_{lat}_{lon}.csv.zip")
    
    if not os.path.exists(pattern): continue
    
    with zipfile.ZipFile(pattern, 'r') as z:
        for csv_name in z.namelist():
            if csv_name.endswith('.csv'):
                df_temp = pd.read_csv(z.open(csv_name))
                df_temp['subbasin_id'] = subid
                dfs_list.append(df_temp)

if not dfs_list:
    print("Nenhum dado encontrado para processar.")
else:
    df_full = pd.concat(dfs_list, ignore_index=True)
    df_full['valid_time'] = pd.to_datetime(df_full['valid_time'])
    df_full['day'] = df_full['valid_time'].dt.date
    
    # Lógica de Agregação Diária
    agg_logic = {
        'u10': ['mean', 'max', 'min'],
        'v10': ['mean', 'max', 'min'],
        'd2m': ['mean', 'max', 'min'],
        't2m': ['mean', 'max', 'min'],
        'ssrd': ['mean', 'max', 'min'],
        'tp': ['sum', 'max', 'min']
    }
    
    print("Processando agregados diários...")
    df_daily = df_full.groupby(['day', 'subbasin_id', 'latitude', 'longitude']).agg(agg_logic)
    df_daily.columns = [f"{col}_{func}" for col, func in df_daily.columns]
    df_daily = df_daily.reset_index()
    
    # Ordenação e Seleção de Colunas
    cols = [
        'day', 'subbasin_id', 'latitude', 'longitude', 
        'u10_mean', 'u10_max', 'u10_min', 'v10_mean', 'v10_max', 'v10_min', 
        'd2m_mean', 'd2m_max', 'd2m_min', 't2m_mean', 't2m_max', 't2m_min', 
        'ssrd_mean', 'ssrd_max', 'ssrd_min', 'tp_sum', 'tp_max', 'tp_min'
    ]
    df_final = df_daily[cols].sort_values(['day', 'subbasin_id'])
    
    output_file = os.path.join(OUTPUT_DIR, f"input_inferencia_{STATION_ID}.csv")
    df_final.to_csv(output_file, index=False)
    
    print(f"✅ Sucesso! Dataset de inferência gerado: {output_file}")
    display(df_final.head())